
# CAPFG 11 — Backbone Robustness with ResNet-50

## Calibration-Aware Probabilistic Feature Generation for Data-Efficient Plant Disease Classification

This notebook evaluates whether the frozen CA-PFG conclusions transfer from DINOv2-Small features to a second backbone:

> **ResNet-50 pretrained on ImageNet, TorchVision `IMAGENET1K_V2` weights**

The robustness experiment preserves the CA-PFG protocol and changes only the visual feature extractor.

### Frozen methods

1. `LR-Real`
2. `PFG-UnweightedLR`
3. `PFG-NW-F1Only`
4. `CA-PFG-MO`

### Frozen task grid

- PlantDoc and PlantVillage
- \(K \in \{5,10,20\}\)
- 10 confirmatory seeds:
  137, 139, 149, 151, 157, 163, 167, 173, 179, 181
- 60 tasks total

### Two operating modes

**Mode A — First run:** attach raw PlantDoc and PlantVillage image datasets.  
The notebook extracts and caches ResNet-50 global average pooled features.

**Mode B — Re-run:** attach the cached feature files created by this notebook.  
Raw image datasets are then optional.

The final ZIP intentionally excludes large feature caches and per-image prediction files.


In [ ]:

# Cell 2 — Imports and environment

from __future__ import annotations

import json
import math
import os
import sys
import hashlib
import shutil
import warnings
import zipfile
from collections import defaultdict
from contextlib import nullcontext
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

import scipy
from scipy.special import logsumexp
from scipy.linalg import solve_triangular

import sklearn
from sklearn.covariance import OAS
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
)
from sklearn.preprocessing import normalize

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models import resnet50, ResNet50_Weights

warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:

# Cell 3 — Frozen robustness configuration

@dataclass(frozen=True)
class CAPFG11Config:
    experiment_version: str = "CAPFG11_resnet50_backbone_robustness"

    # Backbone
    backbone_name: str = "torchvision_resnet50"
    weights_name: str = "IMAGENET1K_V2"
    feature_dimension: int = 2048
    extraction_batch_size: int = 64
    extraction_num_workers: int = 2
    extraction_use_amp: bool = True

    # Frozen task grid
    datasets: Tuple[str, ...] = ("plantdoc", "plantvillage")
    k_values: Tuple[int, ...] = (5, 10, 20)
    confirmatory_seeds: Tuple[int, ...] = (
        137, 139, 149, 151, 157,
        163, 167, 173, 179, 181,
    )

    # Technical smoke
    smoke_dataset: str = "plantdoc"
    smoke_k: int = 5
    smoke_seed: int = 137

    # Frozen representation after backbone
    pca_dimension: int = 64
    pca_whiten: bool = False

    # Frozen probabilistic generator
    pooled_covariance_weight: float = 0.25
    gaussian_jitter: float = 1e-6
    synthetic_multiplier: int = 20
    use_antithetic_sampling: bool = True

    # Frozen classifier/model-selection space
    logreg_c_grid: Tuple[float, ...] = (10.0, 30.0, 100.0)
    synthetic_total_weight_grid: Tuple[float, ...] = (0.125, 0.25, 0.50, 1.00)
    max_iter_logreg: int = 5000

    # Frozen calibration-aware selection
    f1_tolerance: float = 0.002
    nll_tolerance: float = 0.01
    ece_tolerance: float = 0.01
    ece_bins: int = 15

    numerical_tolerance: float = 1e-6


CONFIG = CAPFG11Config()

# First run: keep False and inspect smoke output.
# Then set True and Run All for all 60 tasks.
RUN_FULL_ROBUSTNESS = False

# Feature extraction policy.
# False: reuse cache when exact cache files are found.
# True: force re-extraction from raw images.
FORCE_REEXTRACT_FEATURES = False

# Safety policy for incompatible CUDA devices.
# Recommended: keep False. If the active GPU is not supported by the installed
# PyTorch CUDA build, Notebook 11 fails fast with a clear message instead of
# silently spending many hours extracting 63k+ images on CPU.
ALLOW_CPU_FEATURE_EXTRACTION = False

# Save per-image predictions to disk, but exclude them from final summary ZIP.
SAVE_PREDICTIONS = True

# Optional exact paths. Leave None for auto-discovery.
MANUAL_PATHS = {
    "manifest_split": None,
    "plantdoc_metadata": None,
    "plantvillage_metadata": None,
    "plantdoc_resnet50_features": None,
    "plantvillage_resnet50_features": None,
    "plantdoc_raw_root": None,
    "plantvillage_raw_root": None,
    "resnet50_weights_file": None,
}

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("/mnt/data"),
    Path.cwd(),
]

if Path("/kaggle/working").exists():
    WORK_ROOT = Path("/kaggle/working")
else:
    WORK_ROOT = Path.cwd()

FEATURE_CACHE_DIR = WORK_ROOT / "capfg11_resnet50_feature_cache"
OUTPUT_DIR = WORK_ROOT / "capfg11_outputs"

FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "predictions").mkdir(parents=True, exist_ok=True)

print(CONFIG)
print("Feature cache:", FEATURE_CACHE_DIR)
print("Output directory:", OUTPUT_DIR)
print("RUN_FULL_ROBUSTNESS:", RUN_FULL_ROBUSTNESS)
print("FORCE_REEXTRACT_FEATURES:", FORCE_REEXTRACT_FEATURES)
print("ALLOW_CPU_FEATURE_EXTRACTION:", ALLOW_CPU_FEATURE_EXTRACTION)


In [ ]:

# Cell 4 — Reproducibility and frozen config serialization

def stable_int_hash(value: Any, modulo: int = 2**32 - 1) -> int:
    digest = hashlib.sha256(str(value).encode("utf-8")).hexdigest()
    return int(digest[:16], 16) % modulo


def derive_seed(base_seed: int, component_name: str) -> int:
    return stable_int_hash(f"{int(base_seed)}::{component_name}")


def config_to_jsonable(config: CAPFG11Config) -> Dict[str, Any]:
    return json.loads(json.dumps(asdict(config), default=str))


def stable_hash_config(config: CAPFG11Config) -> str:
    payload = json.dumps(
        config_to_jsonable(config),
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def write_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2, ensure_ascii=False, default=str)


CONFIG_HASH = stable_hash_config(CONFIG)

write_json(
    config_to_jsonable(CONFIG),
    OUTPUT_DIR / "capfg11_frozen_config.json",
)
(OUTPUT_DIR / "capfg11_frozen_config_hash.txt").write_text(
    CONFIG_HASH + "\n",
    encoding="utf-8",
)

print("Configuration SHA-256:", CONFIG_HASH)


In [ ]:

# Cell 5 — File discovery and schema utilities

EXPECTED_FILENAMES = {
    "manifest_split": "manifest_split.csv",
    "plantdoc_metadata": "plantdoc_metadata.csv",
    "plantvillage_metadata": "plantvillage_metadata.csv",
    "plantdoc_resnet50_features": "plantdoc_resnet50_features.npy",
    "plantvillage_resnet50_features": "plantvillage_resnet50_features.npy",
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

ID_CANDIDATES = ("image_id", "sample_id", "id", "uid")
PATH_CANDIDATES = (
    "relative_path", "rel_path", "image_path", "filepath",
    "file_path", "path", "filename", "file_name",
)
LABEL_CANDIDATES = ("label_idx", "class_idx", "class_id", "target", "y", "label")
CLASS_NAME_CANDIDATES = ("class_name", "label_name", "category", "class")
SPLIT_CANDIDATES = ("split", "subset", "partition")
DATASET_CANDIDATES = ("dataset", "dataset_name")


def normalize_dataset_name(value: Any) -> str:
    text = str(value).strip().lower().replace("_", "").replace("-", "").replace(" ", "")
    if "plantdoc" in text:
        return "plantdoc"
    if "plantvillage" in text:
        return "plantvillage"
    return text


def normalize_split_name(value: Any) -> str:
    text = str(value).strip().lower().replace("_", "").replace("-", "").replace(" ", "")
    if text in {"train", "training", "tr"}:
        return "train"
    if text in {"val", "valid", "validation", "dev"}:
        return "validation"
    if text in {"test", "testing", "te"}:
        return "test"
    return text


def normalize_token(value: Any) -> str:
    return "".join(ch.lower() for ch in str(value) if ch.isalnum())


def pick_column(
    frame: pd.DataFrame,
    candidates: Sequence[str],
    required: bool = False,
) -> Optional[str]:
    lower_map = {str(col).strip().lower(): col for col in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    if required:
        raise KeyError(
            f"None of the required columns found: {candidates}. "
            f"Available: {list(frame.columns)}"
        )
    return None


def resolve_named_file(key: str) -> Optional[Path]:
    manual = MANUAL_PATHS.get(key)
    if manual:
        path = Path(manual)
        if not path.exists():
            raise FileNotFoundError(f"Manual path for {key} does not exist: {path}")
        return path

    filename = EXPECTED_FILENAMES[key]
    candidates: List[Path] = []

    # Search input and working locations.
    for root in SEARCH_ROOTS:
        if root.exists():
            candidates.extend(root.rglob(filename))

    # Prefer Kaggle Input cache over working outputs only when unique.
    unique = []
    seen = set()
    for path in candidates:
        resolved = path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        unique.append(path)

    if not unique:
        return None

    unique.sort(
        key=lambda p: (
            0 if "/kaggle/input/" in str(p) else 1,
            len(p.parts),
            len(str(p)),
        )
    )

    if len(unique) > 1:
        print(f"[INFO] Multiple matches for {filename}; using {unique[0]}")
        for alternative in unique[1:4]:
            print("       alternative:", alternative)

    return unique[0]


def create_join_key(frame: pd.DataFrame) -> Tuple[Optional[pd.Series], Optional[str]]:
    id_col = pick_column(frame, ID_CANDIDATES)
    if id_col is not None:
        return frame[id_col].astype(str).str.strip(), f"id:{id_col}"

    path_col = pick_column(frame, PATH_CANDIDATES)
    if path_col is not None:
        key = (
            frame[path_col]
            .astype(str)
            .str.replace("\\\\", "/", regex=False)
            .str.strip()
        )
        return key, f"path:{path_col}"

    return None, None


def harmonize_metadata(
    metadata: pd.DataFrame,
    manifest: pd.DataFrame,
    dataset_key: str,
) -> pd.DataFrame:
    meta = metadata.copy().reset_index(drop=True)
    meta["feature_index"] = np.arange(len(meta), dtype=int)

    man = manifest.copy()
    dataset_col = pick_column(man, DATASET_CANDIDATES)
    if dataset_col is not None:
        man = man.loc[
            man[dataset_col].map(normalize_dataset_name) == dataset_key
        ].copy()

    meta_split_col = pick_column(meta, SPLIT_CANDIDATES)
    meta_label_col = pick_column(meta, LABEL_CANDIDATES)
    meta_class_col = pick_column(meta, CLASS_NAME_CANDIDATES)
    meta_path_col = pick_column(meta, PATH_CANDIDATES)

    # Merge manifest not only when split/label information is missing,
    # but also when metadata lacks an image path. Notebook 2 metadata may
    # already contain labels and splits while the path remains only in
    # manifest_split.csv.
    needs_manifest = (
        meta_split_col is None
        or (meta_label_col is None and meta_class_col is None)
        or meta_path_col is None
    )

    if needs_manifest:
        meta_key, meta_key_desc = create_join_key(meta)
        man_key, man_key_desc = create_join_key(man)

        if meta_key is None or man_key is None:
            raise ValueError(
                "Cannot merge metadata with manifest. "
                "A shared image_id or path-like field is required."
            )

        meta["_join_key"] = meta_key
        man["_join_key"] = man_key

        if meta["_join_key"].duplicated().any():
            raise ValueError(f"Duplicate metadata join keys using {meta_key_desc}.")
        if man["_join_key"].duplicated().any():
            raise ValueError(f"Duplicate manifest join keys using {man_key_desc}.")

        keep_cols = ["_join_key"]
        for group in (
            SPLIT_CANDIDATES,
            LABEL_CANDIDATES,
            CLASS_NAME_CANDIDATES,
            DATASET_CANDIDATES,
            PATH_CANDIDATES,
        ):
            col = pick_column(man, group)
            if col is not None and col not in keep_cols:
                keep_cols.append(col)

        meta = meta.merge(
            man[keep_cols],
            on="_join_key",
            how="left",
            suffixes=("", "_manifest"),
        )

    split_col = pick_column(meta, SPLIT_CANDIDATES)
    if split_col is None:
        split_col = pick_column(
            meta,
            tuple(f"{name}_manifest" for name in SPLIT_CANDIDATES),
        )

    label_col = pick_column(meta, LABEL_CANDIDATES)
    if label_col is None:
        label_col = pick_column(
            meta,
            tuple(f"{name}_manifest" for name in LABEL_CANDIDATES),
        )

    class_col = pick_column(meta, CLASS_NAME_CANDIDATES)
    if class_col is None:
        class_col = pick_column(
            meta,
            tuple(f"{name}_manifest" for name in CLASS_NAME_CANDIDATES),
        )

    if split_col is None:
        raise KeyError("No split column found after harmonization.")

    meta["split_canonical"] = meta[split_col].map(normalize_split_name)

    if label_col is not None:
        numeric = pd.to_numeric(meta[label_col], errors="coerce")
        if numeric.notna().all():
            meta["label_canonical"] = numeric.astype(int)
        else:
            labels_as_text = meta[label_col].astype(str)
            mapping = {
                name: idx
                for idx, name in enumerate(sorted(labels_as_text.unique()))
            }
            meta["label_canonical"] = labels_as_text.map(mapping).astype(int)
    elif class_col is not None:
        names = meta[class_col].astype(str)
        mapping = {name: idx for idx, name in enumerate(sorted(names.unique()))}
        meta["label_canonical"] = names.map(mapping).astype(int)
    else:
        raise KeyError("No label or class-name column found.")

    if class_col is not None:
        meta["class_name_canonical"] = meta[class_col].astype(str)
    else:
        meta["class_name_canonical"] = meta["label_canonical"].astype(str)

    key, _ = create_join_key(meta)
    if key is None:
        meta["sample_key"] = (
            dataset_key + "::" + meta["feature_index"].astype(str)
        )
    else:
        meta["sample_key"] = dataset_key + "::" + key.astype(str)

    valid_splits = {"train", "validation", "test"}
    unknown = sorted(set(meta["split_canonical"]) - valid_splits)
    if unknown:
        raise ValueError(f"Unknown split values: {unknown}")

    if meta["sample_key"].duplicated().any():
        raise ValueError("Duplicate sample_key values detected.")

    # Path availability audit for raw-image feature extraction.
    detected_path_columns = []
    for col in meta.columns:
        low = str(col).strip().lower()
        if (
            low in PATH_CANDIDATES
            or any(low == f"{candidate}_manifest" for candidate in PATH_CANDIDATES)
        ):
            detected_path_columns.append(col)

    if not detected_path_columns:
        raise ValueError(
            f"No path-like column is available after harmonizing {dataset_key}. "
            "The manifest must provide one of: "
            f"{PATH_CANDIDATES}. Available columns: {list(meta.columns)}"
        )

    print(
        f"{dataset_key}: path columns available after harmonization:",
        detected_path_columns,
    )

    return meta


In [ ]:

# Cell 6 — Raw image root discovery and path resolution

def resolve_raw_root(dataset_key: str) -> Optional[Path]:
    manual_key = f"{dataset_key}_raw_root"
    manual = MANUAL_PATHS.get(manual_key)
    if manual:
        path = Path(manual)
        if not path.exists():
            raise FileNotFoundError(f"Raw root does not exist: {path}")
        return path

    directory_names = (
        ("PlantDoc_Cropped", "plantdoc", "PlantDoc")
        if dataset_key == "plantdoc"
        else ("PlantVillage", "plantvillage")
    )

    candidates: List[Path] = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for name in directory_names:
            candidates.extend(
                path
                for path in root.rglob(name)
                if path.is_dir()
            )

    # Score by expected train/test or train/val structure.
    scored = []
    for path in candidates:
        children = {child.name.lower() for child in path.iterdir() if child.is_dir()}
        score = 0
        if "train" in children:
            score += 2
        if dataset_key == "plantdoc" and "test" in children:
            score += 2
        if dataset_key == "plantvillage" and ("val" in children or "validation" in children):
            score += 2
        scored.append((score, -len(path.parts), path))

    if not scored:
        return None

    scored.sort(reverse=True, key=lambda item: (item[0], item[1], str(item[2])))
    selected = scored[0][2]
    print(f"Resolved raw root for {dataset_key}: {selected}")
    return selected


def build_image_index(raw_root: Path) -> Dict[str, List[Path]]:
    index: Dict[str, List[Path]] = defaultdict(list)
    image_count = 0
    for path in raw_root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            index[path.name.lower()].append(path)
            image_count += 1
    print(f"Indexed {image_count:,} images under {raw_root}")
    return dict(index)


def available_path_columns(frame: pd.DataFrame) -> List[str]:
    columns = []
    for col in frame.columns:
        low = str(col).strip().lower()
        if low in PATH_CANDIDATES or any(
            low == f"{candidate}_manifest"
            for candidate in PATH_CANDIDATES
        ):
            columns.append(col)
    return columns


def path_tail_after_marker(text: str, markers: Sequence[str]) -> Optional[Path]:
    normalized = text.replace("\\", "/")
    lower = normalized.lower()
    for marker in markers:
        marker_lower = marker.lower()
        position = lower.find(marker_lower)
        if position >= 0:
            tail = normalized[position + len(marker):].lstrip("/")
            if tail:
                return Path(tail)
    return None


def resolve_row_image_path(
    row: pd.Series,
    dataset_key: str,
    raw_root: Path,
    image_index: Mapping[str, List[Path]],
    path_columns: Sequence[str],
) -> Path:
    markers = (
        ("PlantDoc_Cropped", "plantdoc")
        if dataset_key == "plantdoc"
        else ("PlantVillage", "plantvillage")
    )

    raw_path_values = []
    for col in path_columns:
        value = row.get(col)
        if pd.notna(value):
            text = str(value).strip()
            if text and text.lower() != "nan":
                raw_path_values.append(text)

    # 1) Existing absolute/direct paths.
    for text in raw_path_values:
        direct = Path(text)
        if direct.exists() and direct.is_file():
            return direct

    # 2) Tail after known dataset marker.
    for text in raw_path_values:
        tail = path_tail_after_marker(text, markers)
        if tail is not None:
            candidate = raw_root / tail
            if candidate.exists():
                return candidate

    # 3) Relative path directly under raw root.
    for text in raw_path_values:
        normalized = text.replace("\\", "/").lstrip("/")
        candidate = raw_root / normalized
        if candidate.exists():
            return candidate

    # 4) Basename index with split/class disambiguation.
    basenames = []
    for text in raw_path_values:
        basename = Path(text.replace("\\", "/")).name.lower()
        if basename:
            basenames.append(basename)

    if not basenames:
        # Fall back to filename-like columns that may not have been recognized.
        for col in row.index:
            if "file" in str(col).lower() or "image" in str(col).lower():
                value = row.get(col)
                if pd.notna(value):
                    basename = Path(str(value).replace("\\", "/")).name.lower()
                    if basename:
                        basenames.append(basename)

    split_token = normalize_token(row["split_canonical"])
    class_token = normalize_token(row["class_name_canonical"])

    for basename in basenames:
        candidates = list(image_index.get(basename, []))
        if len(candidates) == 1:
            return candidates[0]
        if len(candidates) > 1:
            scored = []
            for candidate in candidates:
                parts_token = normalize_token("/".join(candidate.parts))
                score = 0
                if class_token and class_token in parts_token:
                    score += 3
                if split_token and split_token in parts_token:
                    score += 1
                # PlantVillage validation split corresponds to raw "val".
                if split_token == "validation" and "val" in parts_token:
                    score += 1
                scored.append((score, candidate))
            scored.sort(reverse=True, key=lambda item: (item[0], str(item[1])))
            if scored[0][0] > 0:
                return scored[0][1]

    raise FileNotFoundError(
        f"Could not resolve image path for sample_key={row.get('sample_key')}. "
        f"Path values={raw_path_values[:3]}"
    )


def resolve_all_image_paths(
    metadata: pd.DataFrame,
    dataset_key: str,
    raw_root: Path,
) -> List[Path]:
    image_index = build_image_index(raw_root)
    path_columns = available_path_columns(metadata)

    resolved = []
    failures = []

    for row_index, row in metadata.iterrows():
        try:
            resolved.append(
                resolve_row_image_path(
                    row,
                    dataset_key,
                    raw_root,
                    image_index,
                    path_columns,
                )
            )
        except Exception as exc:
            failures.append((row_index, row.get("sample_key"), str(exc)))
            if len(failures) >= 10:
                break

    if failures:
        details = "\n".join(str(item) for item in failures)
        raise RuntimeError(
            f"Failed to resolve image paths for {dataset_key}. "
            f"First failures:\n{details}\n"
            "Set MANUAL_PATHS raw_root values or inspect metadata path columns."
        )

    if len(resolved) != len(metadata):
        raise AssertionError("Resolved image path count mismatch.")

    return resolved


In [ ]:

# Cell 7 — Load and harmonize source metadata

def load_source_metadata(dataset_key: str) -> pd.DataFrame:
    manifest_path = resolve_named_file("manifest_split")
    metadata_path = resolve_named_file(f"{dataset_key}_metadata")

    if manifest_path is None:
        raise FileNotFoundError(
            "manifest_split.csv was not found. Attach Notebook 1 output."
        )
    if metadata_path is None:
        raise FileNotFoundError(
            f"{dataset_key}_metadata.csv was not found. "
            "Attach Notebook 2 metadata output."
        )

    manifest = pd.read_csv(manifest_path)
    metadata = pd.read_csv(metadata_path)
    harmonized = harmonize_metadata(metadata, manifest, dataset_key)

    print(f"\n{dataset_key}:")
    print("  manifest:", manifest_path)
    print("  metadata:", metadata_path)
    print("  rows:", len(harmonized))
    print("  classes:", harmonized["label_canonical"].nunique())
    print("  splits:", harmonized["split_canonical"].value_counts().to_dict())

    path_cols = available_path_columns(harmonized)
    print("  path columns:", path_cols)
    if path_cols:
        preview_cols = ["sample_key"] + path_cols[:2]
        display(harmonized[preview_cols].head(3))

    return harmonized


SOURCE_METADATA = {
    dataset_key: load_source_metadata(dataset_key)
    for dataset_key in CONFIG.datasets
}


In [ ]:

# Cell 8 — ResNet-50 feature extraction


def select_feature_extraction_device() -> torch.device:
    """
    Select CUDA only when the active GPU compute capability is explicitly
    supported by the CUDA architectures compiled into the installed PyTorch
    build. torch.cuda.is_available() alone is insufficient for this check.
    """
    if not torch.cuda.is_available():
        if ALLOW_CPU_FEATURE_EXTRACTION:
            print("[DEVICE] CUDA unavailable; using CPU by explicit policy.")
            return torch.device("cpu")
        raise RuntimeError(
            "CUDA is not available. For this large feature-extraction job, "
            "enable a compatible Kaggle GPU, or set "
            "ALLOW_CPU_FEATURE_EXTRACTION=True only if you intentionally "
            "accept CPU extraction."
        )

    try:
        device_name = torch.cuda.get_device_name(0)
        capability = torch.cuda.get_device_capability(0)
        compiled_arches = torch.cuda.get_arch_list()
    except Exception as exc:
        if ALLOW_CPU_FEATURE_EXTRACTION:
            print(
                "[DEVICE] CUDA compatibility inspection failed; "
                "falling back to CPU by explicit policy."
            )
            print("Reason:", repr(exc))
            return torch.device("cpu")
        raise RuntimeError(
            "Could not inspect CUDA device compatibility. "
            "Use a different Kaggle GPU accelerator or set "
            "ALLOW_CPU_FEATURE_EXTRACTION=True to force CPU extraction."
        ) from exc

    capability_tag = f"sm_{capability[0]}{capability[1]}"
    normalized_arches = {str(item).lower() for item in compiled_arches}
    compatible = capability_tag.lower() in normalized_arches

    print("[DEVICE] GPU:", device_name)
    print("[DEVICE] Compute capability:", capability)
    print("[DEVICE] Required architecture tag:", capability_tag)
    print("[DEVICE] PyTorch compiled CUDA architectures:", compiled_arches)

    if compatible:
        print("[DEVICE] CUDA compatibility check: PASSED")
        return torch.device("cuda:0")

    message = (
        f"Incompatible GPU/PyTorch CUDA build: active GPU '{device_name}' "
        f"has compute capability {capability} ({capability_tag}), but the "
        f"installed PyTorch build was compiled for {compiled_arches}. "
        "Switch Kaggle to a GPU whose architecture appears in that list. "
        "Do not continue with this P100 runtime."
    )

    if ALLOW_CPU_FEATURE_EXTRACTION:
        print("[DEVICE] WARNING:", message)
        print("[DEVICE] Falling back to CPU by explicit policy.")
        return torch.device("cpu")

    raise RuntimeError(message)


class ImagePathDataset(Dataset):
    def __init__(
        self,
        image_paths: Sequence[Path],
        transform: Any,
    ) -> None:
        self.image_paths = list(image_paths)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
        path = self.image_paths[index]
        try:
            with Image.open(path) as image:
                image = image.convert("RGB")
                tensor = self.transform(image)
        except Exception as exc:
            raise RuntimeError(f"Failed to load image: {path}") from exc
        return tensor, index


def build_resnet50_feature_extractor(
    device: torch.device,
) -> Tuple[nn.Module, Any]:
    weights = ResNet50_Weights.IMAGENET1K_V2
    manual_weights_file = MANUAL_PATHS.get("resnet50_weights_file")

    if manual_weights_file:
        weights_path = Path(manual_weights_file)
        if not weights_path.exists():
            raise FileNotFoundError(
                f"Manual ResNet-50 weights file does not exist: {weights_path}"
            )

        model = resnet50(weights=None)

        try:
            state_dict = torch.load(
                weights_path,
                map_location="cpu",
                weights_only=True,
            )
        except TypeError:
            state_dict = torch.load(
                weights_path,
                map_location="cpu",
            )

        model.load_state_dict(state_dict)
        print("Loaded ResNet-50 state dict from:", weights_path)
    else:
        try:
            model = resnet50(weights=weights)
        except Exception as exc:
            raise RuntimeError(
                "Could not load/download ResNet50 IMAGENET1K_V2 weights. "
                "Enable Kaggle Internet, use a cached TorchVision checkpoint, "
                "or set MANUAL_PATHS['resnet50_weights_file'] to a compatible "
                "state-dict .pth file."
            ) from exc

    model.fc = nn.Identity()
    model.eval()
    model.to(device)

    # Use the exact preprocessing recipe associated with the selected weights.
    transform = weights.transforms()

    return model, transform


def extract_resnet50_features(
    image_paths: Sequence[Path],
    config: CAPFG11Config,
) -> np.ndarray:
    device = select_feature_extraction_device()
    model, transform = build_resnet50_feature_extractor(device)

    dataset = ImagePathDataset(image_paths, transform)
    loader = DataLoader(
        dataset,
        batch_size=config.extraction_batch_size,
        shuffle=False,
        num_workers=config.extraction_num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(config.extraction_num_workers > 0),
    )

    features = np.empty(
        (len(dataset), config.feature_dimension),
        dtype=np.float32,
    )
    seen = np.zeros(len(dataset), dtype=bool)

    use_amp = config.extraction_use_amp and device.type == "cuda"

    with torch.inference_mode():
        for batch_index, (images, indices) in enumerate(loader):
            images = images.to(device, non_blocking=True)

            context = (
                torch.autocast(device_type="cuda", dtype=torch.float16)
                if use_amp
                else nullcontext()
            )
            with context:
                batch_features = model(images)

            batch_features = batch_features.float().cpu().numpy()
            indices_np = indices.numpy()

            features[indices_np] = batch_features
            seen[indices_np] = True

            if batch_index % 50 == 0 or batch_index == len(loader) - 1:
                print(
                    f"  batch {batch_index + 1}/{len(loader)} "
                    f"({seen.sum():,}/{len(dataset):,} images)"
                )

    if not seen.all():
        missing = np.flatnonzero(~seen)[:10].tolist()
        raise RuntimeError(f"Feature extraction missed indices: {missing}")

    if not np.isfinite(features).all():
        raise ValueError("Extracted features contain NaN or infinite values.")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return features


def cache_paths(dataset_key: str) -> Tuple[Path, Path]:
    return (
        FEATURE_CACHE_DIR / f"{dataset_key}_resnet50_features.npy",
        FEATURE_CACHE_DIR / f"{dataset_key}_resnet50_metadata.csv",
    )


def find_external_feature_cache(dataset_key: str) -> Optional[Path]:
    key = f"{dataset_key}_resnet50_features"
    return resolve_named_file(key)


def load_or_extract_features(
    dataset_key: str,
    metadata: pd.DataFrame,
    config: CAPFG11Config,
) -> Tuple[np.ndarray, pd.DataFrame, Dict[str, Any]]:
    local_feature_path, local_metadata_path = cache_paths(dataset_key)

    if not FORCE_REEXTRACT_FEATURES:
        external_cache = find_external_feature_cache(dataset_key)
        if external_cache is not None:
            features = np.load(external_cache)
            if features.shape != (len(metadata), config.feature_dimension):
                raise ValueError(
                    f"External cache shape mismatch for {dataset_key}: "
                    f"{features.shape}, expected {(len(metadata), config.feature_dimension)}"
                )
            print(f"Loaded cached features for {dataset_key}: {external_cache}")
            return (
                np.asarray(features, dtype=np.float32),
                metadata.copy(),
                {
                    "dataset": dataset_key,
                    "mode": "loaded_cache",
                    "feature_path": str(external_cache),
                    "n_samples": len(metadata),
                    "feature_dimension": config.feature_dimension,
                    "all_finite": bool(np.isfinite(features).all()),
                },
            )

        if local_feature_path.exists():
            features = np.load(local_feature_path)
            if features.shape != (len(metadata), config.feature_dimension):
                raise ValueError(
                    f"Local cache shape mismatch for {dataset_key}: {features.shape}"
                )
            print(f"Loaded local features for {dataset_key}: {local_feature_path}")
            return (
                np.asarray(features, dtype=np.float32),
                metadata.copy(),
                {
                    "dataset": dataset_key,
                    "mode": "loaded_local_cache",
                    "feature_path": str(local_feature_path),
                    "n_samples": len(metadata),
                    "feature_dimension": config.feature_dimension,
                    "all_finite": bool(np.isfinite(features).all()),
                },
            )

    raw_root = resolve_raw_root(dataset_key)
    if raw_root is None:
        raise FileNotFoundError(
            f"No ResNet-50 cache and no raw image root found for {dataset_key}. "
            "Attach the original raw dataset or set MANUAL_PATHS."
        )

    image_paths = resolve_all_image_paths(metadata, dataset_key, raw_root)

    # Fast image-path smoke before long extraction.
    for path in image_paths[:5]:
        with Image.open(path) as image:
            image.verify()

    print(f"Extracting ResNet-50 features for {dataset_key}: {len(image_paths):,} images")
    features = extract_resnet50_features(image_paths, config)

    np.save(local_feature_path, features)
    metadata_to_save = metadata.copy()
    metadata_to_save["resolved_image_path"] = [str(path) for path in image_paths]
    metadata_to_save.to_csv(local_metadata_path, index=False)

    return (
        features,
        metadata,
        {
            "dataset": dataset_key,
            "mode": "extracted_from_raw_images",
            "feature_path": str(local_feature_path),
            "metadata_path": str(local_metadata_path),
            "raw_root": str(raw_root),
            "n_samples": len(metadata),
            "feature_dimension": config.feature_dimension,
            "all_finite": bool(np.isfinite(features).all()),
        },
    )


FEATURE_BUNDLES_RAW = {}
FEATURE_AUDIT_ROWS = []

for dataset_key in CONFIG.datasets:
    features, metadata, audit = load_or_extract_features(
        dataset_key,
        SOURCE_METADATA[dataset_key],
        CONFIG,
    )
    FEATURE_BUNDLES_RAW[dataset_key] = (features, metadata)
    FEATURE_AUDIT_ROWS.append(audit)

feature_audit = pd.DataFrame(FEATURE_AUDIT_ROWS)
feature_audit.to_csv(
    OUTPUT_DIR / "feature_extraction_audit.csv",
    index=False,
)
display(feature_audit)


In [ ]:

# Cell 9 — Feature-bundle validation

@dataclass
class DatasetBundle:
    dataset: str
    features: np.ndarray
    metadata: pd.DataFrame
    class_labels: np.ndarray


BUNDLES: Dict[str, DatasetBundle] = {}

for dataset_key, (features, metadata) in FEATURE_BUNDLES_RAW.items():
    if features.ndim != 2:
        raise ValueError(f"{dataset_key}: feature matrix must be 2-D.")
    if len(features) != len(metadata):
        raise ValueError(
            f"{dataset_key}: feature/metadata length mismatch "
            f"{len(features)} vs {len(metadata)}"
        )
    if features.shape[1] != CONFIG.feature_dimension:
        raise ValueError(
            f"{dataset_key}: expected feature dimension "
            f"{CONFIG.feature_dimension}, got {features.shape[1]}"
        )
    if not np.isfinite(features).all():
        raise ValueError(f"{dataset_key}: non-finite feature values.")

    class_labels = np.sort(metadata["label_canonical"].unique())

    BUNDLES[dataset_key] = DatasetBundle(
        dataset=dataset_key,
        features=np.asarray(features, dtype=np.float32),
        metadata=metadata.reset_index(drop=True),
        class_labels=class_labels,
    )

    print(
        dataset_key,
        "shape=", features.shape,
        "classes=", len(class_labels),
        "splits=", metadata["split_canonical"].value_counts().to_dict(),
    )

print("Feature-bundle validation: PASSED")


In [ ]:

# Cell 10 — Task split construction and support-only preprocessing

@dataclass
class TaskSplit:
    support_indices: np.ndarray
    validation_indices: np.ndarray
    test_indices: np.ndarray
    class_labels: np.ndarray


@dataclass
class SupportPreprocessor:
    pca: PCA
    n_components: int


def build_task_split(
    bundle: DatasetBundle,
    k: int,
    seed: int,
) -> TaskSplit:
    meta = bundle.metadata
    train = meta.loc[meta["split_canonical"] == "train"]

    validation_indices = meta.index[
        meta["split_canonical"] == "validation"
    ].to_numpy(dtype=int)

    test_indices = meta.index[
        meta["split_canonical"] == "test"
    ].to_numpy(dtype=int)

    support_parts = []

    for label in bundle.class_labels:
        candidates = train.index[
            train["label_canonical"] == label
        ].to_numpy(dtype=int)

        if len(candidates) < k:
            raise ValueError(
                f"{bundle.dataset}: class {label} has only "
                f"{len(candidates)} train samples, K={k}."
            )

        rng = np.random.default_rng(
            derive_seed(seed, f"support::{bundle.dataset}::{k}::{label}")
        )
        chosen = np.sort(
            rng.choice(candidates, size=k, replace=False)
        )
        support_parts.append(chosen)

    support_indices = np.sort(np.concatenate(support_parts))

    support_set = set(support_indices.tolist())
    validation_set = set(validation_indices.tolist())
    test_set = set(test_indices.tolist())

    if support_set & validation_set:
        raise AssertionError("Support/validation overlap detected.")
    if support_set & test_set:
        raise AssertionError("Support/test overlap detected.")
    if validation_set & test_set:
        raise AssertionError("Validation/test overlap detected.")

    support_labels = meta.loc[
        support_indices,
        "label_canonical",
    ].to_numpy(dtype=int)

    observed, counts = np.unique(support_labels, return_counts=True)

    if not np.array_equal(observed, bundle.class_labels):
        raise AssertionError("Support does not cover all classes.")
    if not np.all(counts == k):
        raise AssertionError("Support is not exactly K-shot per class.")

    return TaskSplit(
        support_indices=support_indices,
        validation_indices=validation_indices,
        test_indices=test_indices,
        class_labels=bundle.class_labels.copy(),
    )


def fit_support_preprocessor(
    X_support_raw: np.ndarray,
    config: CAPFG11Config,
    seed: int,
) -> SupportPreprocessor:
    X_support_l2 = normalize(X_support_raw, norm="l2")

    n_components = min(
        config.pca_dimension,
        X_support_l2.shape[0] - 1,
        X_support_l2.shape[1],
    )

    if n_components < 2:
        raise ValueError(f"PCA dimension too small: {n_components}")

    pca = PCA(
        n_components=n_components,
        whiten=config.pca_whiten,
        svd_solver="full",
        random_state=seed,
    )
    pca.fit(X_support_l2)

    return SupportPreprocessor(
        pca=pca,
        n_components=n_components,
    )


def transform_with_support_preprocessor(
    preprocessor: SupportPreprocessor,
    X_raw: np.ndarray,
) -> np.ndarray:
    X_l2 = normalize(X_raw, norm="l2")
    X_pca = preprocessor.pca.transform(X_l2)

    if not np.isfinite(X_pca).all():
        raise ValueError("Non-finite transformed features.")

    return np.asarray(X_pca, dtype=np.float64)


In [ ]:

# Cell 11 — Stable Gaussian estimation

@dataclass
class GaussianModel:
    class_labels: np.ndarray
    means: np.ndarray
    class_covariances: np.ndarray
    base_covariances: np.ndarray
    pooled_covariance: np.ndarray
    oas_shrinkages: np.ndarray
    class_counts: np.ndarray


def ensure_positive_definite(
    covariance: np.ndarray,
    jitter: float,
) -> np.ndarray:
    covariance = np.asarray(covariance, dtype=np.float64)
    covariance = 0.5 * (covariance + covariance.T)

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    floor = max(float(jitter), np.finfo(np.float64).eps)
    eigenvalues = np.maximum(eigenvalues, floor)

    repaired = (eigenvectors * eigenvalues) @ eigenvectors.T
    return 0.5 * (repaired + repaired.T)


def fit_oas_covariance(
    X: np.ndarray,
    jitter: float,
    assume_centered: bool = False,
) -> Tuple[np.ndarray, float]:
    estimator = OAS(
        assume_centered=assume_centered,
        store_precision=False,
    )
    estimator.fit(np.asarray(X, dtype=np.float64))

    covariance = ensure_positive_definite(
        estimator.covariance_,
        jitter,
    )

    return covariance, float(estimator.shrinkage_)


def estimate_gaussian_model(
    X_support: np.ndarray,
    y_support: np.ndarray,
    class_labels: np.ndarray,
    pooled_weight: float,
    jitter: float,
) -> GaussianModel:
    means = []
    class_covariances = []
    shrinkages = []
    counts = []
    residuals = []

    for label in class_labels:
        Xc = X_support[y_support == label]

        if len(Xc) < 2:
            raise ValueError(
                f"At least two support samples are required for class {label}."
            )

        mean_c = Xc.mean(axis=0)
        covariance_c, shrinkage_c = fit_oas_covariance(
            Xc,
            jitter,
            assume_centered=False,
        )

        means.append(mean_c)
        class_covariances.append(covariance_c)
        shrinkages.append(shrinkage_c)
        counts.append(len(Xc))
        residuals.append(Xc - mean_c)

    residual_matrix = np.vstack(residuals)

    pooled_covariance, _ = fit_oas_covariance(
        residual_matrix,
        jitter,
        assume_centered=True,
    )

    base_covariances = []

    for covariance_c in class_covariances:
        base = (
            (1.0 - pooled_weight) * covariance_c
            + pooled_weight * pooled_covariance
        )
        base_covariances.append(
            ensure_positive_definite(base, jitter)
        )

    return GaussianModel(
        class_labels=np.asarray(class_labels, dtype=int),
        means=np.asarray(means, dtype=np.float64),
        class_covariances=np.asarray(class_covariances, dtype=np.float64),
        base_covariances=np.asarray(base_covariances, dtype=np.float64),
        pooled_covariance=np.asarray(pooled_covariance, dtype=np.float64),
        oas_shrinkages=np.asarray(shrinkages, dtype=np.float64),
        class_counts=np.asarray(counts, dtype=int),
    )


In [ ]:

# Cell 12 — Antithetic probabilistic feature generation

@dataclass
class SyntheticBatch:
    features: np.ndarray
    labels: np.ndarray
    source_positions: np.ndarray
    pair_ids: np.ndarray


def generate_antithetic_features(
    gaussian_model: GaussianModel,
    samples_per_class: int,
    task_seed: int,
    use_antithetic: bool = True,
) -> SyntheticBatch:
    all_features = []
    all_labels = []
    all_source_positions = []
    all_pair_ids = []
    next_pair_id = 0

    for class_position, label in enumerate(gaussian_model.class_labels):
        rng = np.random.default_rng(
            derive_seed(task_seed, f"synthetic_noise::{label}")
        )

        mean = gaussian_model.means[class_position]
        covariance = gaussian_model.base_covariances[class_position]
        chol = np.linalg.cholesky(covariance)
        dimension = mean.shape[0]

        if use_antithetic:
            n_pairs = samples_per_class // 2
            epsilon = rng.standard_normal((n_pairs, dimension))
            deviations = epsilon @ chol.T

            generated = np.vstack([
                mean + deviations,
                mean - deviations,
            ])

            pair_ids = np.concatenate([
                np.arange(next_pair_id, next_pair_id + n_pairs),
                np.arange(next_pair_id, next_pair_id + n_pairs),
            ])

            next_pair_id += n_pairs

            if len(generated) < samples_per_class:
                extra_epsilon = rng.standard_normal((1, dimension))
                extra = mean + extra_epsilon @ chol.T
                generated = np.vstack([generated, extra])
                pair_ids = np.concatenate([
                    pair_ids,
                    np.array([next_pair_id]),
                ])
                next_pair_id += 1
        else:
            epsilon = rng.standard_normal(
                (samples_per_class, dimension)
            )
            generated = mean + epsilon @ chol.T
            pair_ids = np.arange(
                next_pair_id,
                next_pair_id + samples_per_class,
            )
            next_pair_id += samples_per_class

        generated = generated[:samples_per_class]
        pair_ids = pair_ids[:samples_per_class]

        all_features.append(generated)
        all_labels.append(
            np.full(samples_per_class, int(label), dtype=int)
        )
        all_source_positions.append(
            np.full(samples_per_class, class_position, dtype=int)
        )
        all_pair_ids.append(pair_ids.astype(int))

    X_syn = np.vstack(all_features).astype(np.float64)
    y_syn = np.concatenate(all_labels)

    if not np.isfinite(X_syn).all():
        raise ValueError("Synthetic features contain non-finite values.")

    return SyntheticBatch(
        features=X_syn,
        labels=y_syn,
        source_positions=np.concatenate(all_source_positions),
        pair_ids=np.concatenate(all_pair_ids),
    )


def normalize_synthetic_weights(
    n_synthetic: int,
    lambda_syn: float,
    n_real: int,
    tolerance: float,
) -> np.ndarray:
    budget = float(lambda_syn) * int(n_real)
    weights = np.full(
        n_synthetic,
        budget / n_synthetic,
        dtype=np.float64,
    )

    if abs(weights.sum() - budget) > tolerance:
        raise AssertionError(
            f"Synthetic weight budget mismatch: {weights.sum()} vs {budget}"
        )

    return weights


In [ ]:

# Cell 13 — Metrics

def multiclass_brier_score(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    class_labels: np.ndarray,
) -> float:
    label_to_position = {
        int(label): position
        for position, label in enumerate(class_labels)
    }

    one_hot = np.zeros_like(probabilities, dtype=np.float64)
    positions = np.array(
        [label_to_position[int(label)] for label in y_true],
        dtype=int,
    )
    one_hot[np.arange(len(y_true)), positions] = 1.0

    return float(
        np.mean(np.sum((probabilities - one_hot) ** 2, axis=1))
    )


def expected_calibration_error(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    class_labels: np.ndarray,
    n_bins: int = 15,
) -> float:
    predicted_positions = probabilities.argmax(axis=1)
    predicted_labels = class_labels[predicted_positions]
    confidence = probabilities.max(axis=1)
    correct = (predicted_labels == y_true).astype(np.float64)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for bin_index in range(n_bins):
        lower = bin_edges[bin_index]
        upper = bin_edges[bin_index + 1]

        if bin_index == 0:
            mask = (confidence >= lower) & (confidence <= upper)
        else:
            mask = (confidence > lower) & (confidence <= upper)

        if not mask.any():
            continue

        bin_accuracy = correct[mask].mean()
        bin_confidence = confidence[mask].mean()

        ece += mask.mean() * abs(bin_accuracy - bin_confidence)

    return float(ece)


def evaluate_probabilities(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    class_labels: np.ndarray,
    ece_bins: int,
) -> Dict[str, float]:
    predicted_labels = class_labels[probabilities.argmax(axis=1)]

    return {
        "macro_f1": float(
            f1_score(
                y_true,
                predicted_labels,
                labels=class_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "accuracy": float(
            accuracy_score(y_true, predicted_labels)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, predicted_labels)
        ),
        "nll": float(
            log_loss(y_true, probabilities, labels=class_labels)
        ),
        "ece": expected_calibration_error(
            y_true,
            probabilities,
            class_labels,
            n_bins=ece_bins,
        ),
        "brier": multiclass_brier_score(
            y_true,
            probabilities,
            class_labels,
        ),
    }


In [ ]:

# Cell 14 — Classifier fitting and frozen selectors

@dataclass
class CandidateModel:
    method: str
    C: float
    lambda_syn: Optional[float]
    validation_metrics: Dict[str, float]
    model: LogisticRegression


def fit_logistic_regression(
    X_real: np.ndarray,
    y_real: np.ndarray,
    C: float,
    seed: int,
    max_iter: int,
    X_syn: Optional[np.ndarray] = None,
    y_syn: Optional[np.ndarray] = None,
    synthetic_weights: Optional[np.ndarray] = None,
) -> LogisticRegression:
    if X_syn is None:
        X_train = X_real
        y_train = y_real
        sample_weight = np.ones(len(y_real), dtype=np.float64)
    else:
        if y_syn is None:
            raise ValueError("y_syn is required when X_syn is provided.")

        X_train = np.vstack([X_real, X_syn])
        y_train = np.concatenate([y_real, y_syn])

        if synthetic_weights is None:
            synthetic_weights = np.ones(
                len(y_syn),
                dtype=np.float64,
            )

        sample_weight = np.concatenate([
            np.ones(len(y_real), dtype=np.float64),
            np.asarray(synthetic_weights, dtype=np.float64),
        ])

    model = LogisticRegression(
        C=float(C),
        solver="lbfgs",
        max_iter=int(max_iter),
        random_state=int(seed),
    )
    model.fit(
        X_train,
        y_train,
        sample_weight=sample_weight,
    )

    return model


def aligned_predict_proba(
    model: LogisticRegression,
    X: np.ndarray,
    class_labels: np.ndarray,
) -> np.ndarray:
    raw = model.predict_proba(X)

    aligned = np.zeros(
        (len(X), len(class_labels)),
        dtype=np.float64,
    )

    position = {
        int(label): idx
        for idx, label in enumerate(class_labels)
    }

    for model_position, label in enumerate(model.classes_):
        aligned[:, position[int(label)]] = raw[:, model_position]

    aligned = np.clip(aligned, 1e-15, 1.0)
    aligned /= aligned.sum(axis=1, keepdims=True)

    return aligned


def candidate_sort_key(
    candidate: CandidateModel,
) -> Tuple[float, float]:
    lambda_value = (
        float("inf")
        if candidate.lambda_syn is None
        else float(candidate.lambda_syn)
    )
    return lambda_value, float(candidate.C)


def select_f1_only(
    candidates: Sequence[CandidateModel],
) -> CandidateModel:
    best_f1 = max(
        candidate.validation_metrics["macro_f1"]
        for candidate in candidates
    )

    stage = [
        candidate
        for candidate in candidates
        if candidate.validation_metrics["macro_f1"]
        >= best_f1 - 1e-12
    ]

    return sorted(stage, key=candidate_sort_key)[0]


def select_multiobjective(
    candidates: Sequence[CandidateModel],
    config: CAPFG11Config,
) -> CandidateModel:
    best_f1 = max(
        candidate.validation_metrics["macro_f1"]
        for candidate in candidates
    )

    stage_f1 = [
        candidate
        for candidate in candidates
        if candidate.validation_metrics["macro_f1"]
        >= best_f1 - config.f1_tolerance
    ]

    best_nll = min(
        candidate.validation_metrics["nll"]
        for candidate in stage_f1
    )

    stage_nll = [
        candidate
        for candidate in stage_f1
        if candidate.validation_metrics["nll"]
        <= best_nll + config.nll_tolerance
    ]

    best_ece = min(
        candidate.validation_metrics["ece"]
        for candidate in stage_nll
    )

    stage_ece = [
        candidate
        for candidate in stage_nll
        if candidate.validation_metrics["ece"]
        <= best_ece + config.ece_tolerance
    ]

    return sorted(stage_ece, key=candidate_sort_key)[0]


In [ ]:

# Cell 15 — Robustness task engine

def make_task_key(dataset: str, k: int, seed: int) -> str:
    return f"{dataset}__k{k}__seed{seed}"


def prediction_frame(
    task_key: str,
    dataset: str,
    k: int,
    seed: int,
    method: str,
    split_name: str,
    sample_keys: Sequence[str],
    y_true: np.ndarray,
    probabilities: np.ndarray,
    class_labels: np.ndarray,
) -> pd.DataFrame:
    predicted_labels = class_labels[
        probabilities.argmax(axis=1)
    ]

    return pd.DataFrame({
        "task_key": task_key,
        "dataset": dataset,
        "K": k,
        "seed": seed,
        "method": method,
        "split": split_name,
        "sample_key": list(sample_keys),
        "true_label": y_true.astype(int),
        "predicted_label": predicted_labels.astype(int),
        "probability_vector": [
            json.dumps(row.tolist(), separators=(",", ":"))
            for row in probabilities
        ],
    })


def evaluate_selected_candidate(
    candidate: CandidateModel,
    X: np.ndarray,
    y: np.ndarray,
    class_labels: np.ndarray,
    config: CAPFG11Config,
) -> Tuple[np.ndarray, Dict[str, float]]:
    probabilities = aligned_predict_proba(
        candidate.model,
        X,
        class_labels,
    )

    metrics = evaluate_probabilities(
        y,
        probabilities,
        class_labels,
        config.ece_bins,
    )

    return probabilities, metrics


def run_robustness_task(
    bundle: DatasetBundle,
    k: int,
    seed: int,
    config: CAPFG11Config,
) -> Dict[str, pd.DataFrame]:
    task_key = make_task_key(bundle.dataset, k, seed)
    meta = bundle.metadata

    split = build_task_split(bundle, k, seed)

    support_idx = split.support_indices
    validation_idx = split.validation_indices
    test_idx = split.test_indices

    X_support_raw = bundle.features[support_idx]
    X_validation_raw = bundle.features[validation_idx]
    X_test_raw = bundle.features[test_idx]

    y_support = meta.loc[
        support_idx,
        "label_canonical",
    ].to_numpy(dtype=int)

    y_validation = meta.loc[
        validation_idx,
        "label_canonical",
    ].to_numpy(dtype=int)

    y_test = meta.loc[
        test_idx,
        "label_canonical",
    ].to_numpy(dtype=int)

    # Support-only preprocessing.
    preprocessor = fit_support_preprocessor(
        X_support_raw,
        config,
        seed=derive_seed(seed, "pca"),
    )

    X_support = transform_with_support_preprocessor(
        preprocessor,
        X_support_raw,
    )
    X_validation = transform_with_support_preprocessor(
        preprocessor,
        X_validation_raw,
    )
    X_test = transform_with_support_preprocessor(
        preprocessor,
        X_test_raw,
    )

    # Support-only probabilistic generator.
    gaussian_model = estimate_gaussian_model(
        X_support,
        y_support,
        split.class_labels,
        pooled_weight=config.pooled_covariance_weight,
        jitter=config.gaussian_jitter,
    )

    samples_per_class = config.synthetic_multiplier * k

    synthetic = generate_antithetic_features(
        gaussian_model,
        samples_per_class,
        task_seed=seed,
        use_antithetic=config.use_antithetic_sampling,
    )

    all_candidates = []
    selected: Dict[str, CandidateModel] = {}

    # 1) LR-Real.
    lr_candidates = []

    for C in config.logreg_c_grid:
        model = fit_logistic_regression(
            X_support,
            y_support,
            C=C,
            seed=derive_seed(seed, f"lr_real::{C}"),
            max_iter=config.max_iter_logreg,
        )

        val_prob = aligned_predict_proba(
            model,
            X_validation,
            split.class_labels,
        )

        candidate = CandidateModel(
            method="LR-Real",
            C=C,
            lambda_syn=None,
            validation_metrics=evaluate_probabilities(
                y_validation,
                val_prob,
                split.class_labels,
                config.ece_bins,
            ),
            model=model,
        )

        lr_candidates.append(candidate)
        all_candidates.append(candidate)

    selected["LR-Real"] = select_f1_only(lr_candidates)

    # 2) PFG-UnweightedLR.
    unweighted_candidates = []
    unit_weights = np.ones(
        len(synthetic.features),
        dtype=np.float64,
    )

    for C in config.logreg_c_grid:
        model = fit_logistic_regression(
            X_support,
            y_support,
            C=C,
            seed=derive_seed(seed, f"pfg_unweighted::{C}"),
            max_iter=config.max_iter_logreg,
            X_syn=synthetic.features,
            y_syn=synthetic.labels,
            synthetic_weights=unit_weights,
        )

        val_prob = aligned_predict_proba(
            model,
            X_validation,
            split.class_labels,
        )

        candidate = CandidateModel(
            method="PFG-UnweightedLR",
            C=C,
            lambda_syn=None,
            validation_metrics=evaluate_probabilities(
                y_validation,
                val_prob,
                split.class_labels,
                config.ece_bins,
            ),
            model=model,
        )

        unweighted_candidates.append(candidate)
        all_candidates.append(candidate)

    selected["PFG-UnweightedLR"] = select_f1_only(
        unweighted_candidates
    )

    # 3/4) One normalized-budget candidate pool, two selectors.
    normalized_candidates = []

    for lambda_syn in config.synthetic_total_weight_grid:
        weights = normalize_synthetic_weights(
            len(synthetic.features),
            lambda_syn,
            len(X_support),
            config.numerical_tolerance,
        )

        for C in config.logreg_c_grid:
            model = fit_logistic_regression(
                X_support,
                y_support,
                C=C,
                seed=derive_seed(
                    seed,
                    f"pfg_nw::{lambda_syn}::{C}",
                ),
                max_iter=config.max_iter_logreg,
                X_syn=synthetic.features,
                y_syn=synthetic.labels,
                synthetic_weights=weights,
            )

            val_prob = aligned_predict_proba(
                model,
                X_validation,
                split.class_labels,
            )

            candidate = CandidateModel(
                method="PFG-NW",
                C=C,
                lambda_syn=lambda_syn,
                validation_metrics=evaluate_probabilities(
                    y_validation,
                    val_prob,
                    split.class_labels,
                    config.ece_bins,
                ),
                model=model,
            )

            normalized_candidates.append(candidate)
            all_candidates.append(candidate)

    selected["PFG-NW-F1Only"] = select_f1_only(
        normalized_candidates
    )
    selected["CA-PFG-MO"] = select_multiobjective(
        normalized_candidates,
        config,
    )

    # Test evaluation only after selection.
    task_rows = []
    prediction_frames = []

    for method_name, candidate in selected.items():
        val_prob, val_metrics = evaluate_selected_candidate(
            candidate,
            X_validation,
            y_validation,
            split.class_labels,
            config,
        )

        test_prob, test_metrics = evaluate_selected_candidate(
            candidate,
            X_test,
            y_test,
            split.class_labels,
            config,
        )

        expected_budget = (
            0.0
            if candidate.lambda_syn is None
            else float(candidate.lambda_syn) * len(X_support)
        )

        row = {
            "task_key": task_key,
            "dataset": bundle.dataset,
            "K": k,
            "seed": seed,
            "backbone": config.backbone_name,
            "weights_name": config.weights_name,
            "method": method_name,
            "selected_C": float(candidate.C),
            "selected_lambda_syn": candidate.lambda_syn,
            "expected_synthetic_budget": expected_budget,
            "pca_components": preprocessor.n_components,
            "config_hash": CONFIG_HASH,
        }

        row.update({
            f"validation_{key}": value
            for key, value in val_metrics.items()
        })
        row.update({
            f"test_{key}": value
            for key, value in test_metrics.items()
        })

        task_rows.append(row)

        if SAVE_PREDICTIONS:
            for split_name, indices, y_true, probabilities in (
                (
                    "validation",
                    validation_idx,
                    y_validation,
                    val_prob,
                ),
                (
                    "test",
                    test_idx,
                    y_test,
                    test_prob,
                ),
            ):
                prediction_frames.append(
                    prediction_frame(
                        task_key=task_key,
                        dataset=bundle.dataset,
                        k=k,
                        seed=seed,
                        method=method_name,
                        split_name=split_name,
                        sample_keys=meta.loc[
                            indices,
                            "sample_key",
                        ].tolist(),
                        y_true=y_true,
                        probabilities=probabilities,
                        class_labels=split.class_labels,
                    )
                )

    candidate_rows = []

    selected_ids = {
        id(candidate): method_name
        for method_name, candidate in selected.items()
    }

    for candidate in all_candidates:
        record = {
            "task_key": task_key,
            "dataset": bundle.dataset,
            "K": k,
            "seed": seed,
            "method_pool": candidate.method,
            "C": candidate.C,
            "lambda_syn": candidate.lambda_syn,
            "selected_as": selected_ids.get(id(candidate), ""),
        }
        record.update({
            f"validation_{key}": value
            for key, value in candidate.validation_metrics.items()
        })
        candidate_rows.append(record)

    support_keys = set(meta.loc[support_idx, "sample_key"])
    validation_keys = set(meta.loc[validation_idx, "sample_key"])
    test_keys = set(meta.loc[test_idx, "sample_key"])

    protocol_checks = {
        "support_validation_disjoint": len(
            support_keys & validation_keys
        ) == 0,
        "support_test_disjoint": len(
            support_keys & test_keys
        ) == 0,
        "validation_test_disjoint": len(
            validation_keys & test_keys
        ) == 0,
        "support_exact_k_per_class": bool(
            np.all(
                np.unique(
                    y_support,
                    return_counts=True,
                )[1]
                == k
            )
        ),
        "pca_fitted_on_support_count": int(
            preprocessor.pca.n_samples_
        )
        == len(X_support),
        "gaussian_estimation_support_only": True,
        "synthetic_generation_support_only": True,
        "validation_for_selection_only": True,
        "test_after_selection_only": True,
        "config_hash_match": (
            CONFIG_HASH == stable_hash_config(CONFIG)
        ),
    }

    overall_pass = all(protocol_checks.values())

    protocol_row = {
        "task_key": task_key,
        "dataset": bundle.dataset,
        "K": k,
        "seed": seed,
        "backbone": config.backbone_name,
        **protocol_checks,
        "overall_protocol_pass": overall_pass,
        "config_hash": CONFIG_HASH,
    }

    if not overall_pass:
        failed = [
            key
            for key, value in protocol_checks.items()
            if not value
        ]
        raise AssertionError(
            f"{task_key}: protocol audit failed: {failed}"
        )

    return {
        "task_results": pd.DataFrame(task_rows),
        "candidate_results": pd.DataFrame(candidate_rows),
        "protocol_audit": pd.DataFrame([protocol_row]),
        "predictions": (
            pd.concat(prediction_frames, ignore_index=True)
            if prediction_frames
            else pd.DataFrame()
        ),
    }


In [ ]:

# Cell 16 — Incremental export helpers

def safe_write_table(
    frame: pd.DataFrame,
    parquet_path: Path,
) -> Path:
    parquet_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        frame.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as exc:
        csv_path = parquet_path.with_suffix(".csv")
        print(
            f"[WARN] Parquet export failed ({exc}); "
            f"writing CSV: {csv_path}"
        )
        frame.to_csv(csv_path, index=False)
        return csv_path


def save_task_predictions(
    output: Dict[str, pd.DataFrame],
    task_key: str,
) -> Optional[Path]:
    if not SAVE_PREDICTIONS or len(output["predictions"]) == 0:
        return None

    return safe_write_table(
        output["predictions"],
        OUTPUT_DIR / "predictions" / f"{task_key}.parquet",
    )


def small_view(
    output: Dict[str, pd.DataFrame],
) -> Dict[str, pd.DataFrame]:
    return {
        "task_results": output["task_results"],
        "candidate_results": output["candidate_results"],
        "protocol_audit": output["protocol_audit"],
    }


def collect_small_table(
    outputs: Sequence[Dict[str, pd.DataFrame]],
    key: str,
) -> pd.DataFrame:
    frames = [
        output[key]
        for output in outputs
        if key in output and len(output[key])
    ]

    return (
        pd.concat(frames, ignore_index=True)
        if frames
        else pd.DataFrame()
    )


def save_small_bundle(
    outputs: Sequence[Dict[str, pd.DataFrame]],
    prefix: str,
) -> Dict[str, Path]:
    mappings = {
        "task_results": f"{prefix}_task_results.csv",
        "candidate_results": f"{prefix}_candidate_results.csv",
        "protocol_audit": f"{prefix}_protocol_audit.csv",
    }

    saved = {}

    for key, filename in mappings.items():
        frame = collect_small_table(outputs, key)
        path = OUTPUT_DIR / filename
        frame.to_csv(path, index=False)
        saved[key] = path

    return saved


def create_summary_zip(
    zip_name: str = "CAPFG11_backbone_robustness_summary_package.zip",
) -> Path:
    archive_path = WORK_ROOT / zip_name

    if archive_path.exists():
        archive_path.unlink()

    include_suffixes = {".csv", ".json", ".txt"}

    with zipfile.ZipFile(
        archive_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as archive:
        for path in OUTPUT_DIR.rglob("*"):
            if not path.is_file():
                continue
            if "predictions" in path.parts:
                continue
            if path.suffix.lower() not in include_suffixes:
                continue

            archive.write(
                path,
                arcname=path.relative_to(OUTPUT_DIR),
            )

    return archive_path



## Technical smoke test

Run one task first:

- PlantDoc
- \(K=5\)
- seed 137

The smoke run verifies the full ResNet-50 feature-space CA-PFG pipeline before the 60-task robustness evaluation.


In [ ]:

# Cell 18 — Run technical smoke test

SMOKE_OUTPUT = run_robustness_task(
    BUNDLES[CONFIG.smoke_dataset],
    k=CONFIG.smoke_k,
    seed=CONFIG.smoke_seed,
    config=CONFIG,
)

SMOKE_TASK_KEY = make_task_key(
    CONFIG.smoke_dataset,
    CONFIG.smoke_k,
    CONFIG.smoke_seed,
)

save_task_predictions(
    SMOKE_OUTPUT,
    SMOKE_TASK_KEY,
)

SMOKE_SMALL = small_view(SMOKE_OUTPUT)
save_small_bundle([SMOKE_SMALL], prefix="smoke")

print("Smoke task:", SMOKE_TASK_KEY)

display(
    SMOKE_OUTPUT["task_results"]
    .sort_values("method")
    .reset_index(drop=True)
)

display(SMOKE_OUTPUT["protocol_audit"].T)

SMOKE_READY_FOR_FULL = bool(
    SMOKE_OUTPUT["protocol_audit"]
    .iloc[0]["overall_protocol_pass"]
)

print("SMOKE_READY_FOR_FULL:", SMOKE_READY_FOR_FULL)


In [ ]:

# Cell 19 — Smoke comparison

smoke_results = SMOKE_OUTPUT["task_results"].copy()

display(
    smoke_results[
        [
            "method",
            "selected_C",
            "selected_lambda_syn",
            "test_macro_f1",
            "test_nll",
            "test_ece",
            "test_brier",
        ]
    ]
    .sort_values("method")
    .reset_index(drop=True)
)

fig = plt.figure(figsize=(8, 5))

plt.scatter(
    smoke_results["test_macro_f1"],
    smoke_results["test_ece"],
)

for _, row in smoke_results.iterrows():
    plt.annotate(
        row["method"],
        (row["test_macro_f1"], row["test_ece"]),
    )

plt.xlabel("Test Macro-F1")
plt.ylabel("Test ECE")
plt.title("ResNet-50 smoke: discrimination–calibration trade-off")
plt.tight_layout()
plt.show()



## Full backbone robustness run

After smoke audit passes, set:

```python
RUN_FULL_ROBUSTNESS = True
```

and Run All.

The full experiment contains:

\[
2 \times 3 \times 10 = 60
\]

tasks and evaluates the same frozen CA-PFG logic under ResNet-50 features.


In [ ]:

# Cell 21 — Run 60 robustness tasks

ROBUSTNESS_SMALL_OUTPUTS: List[Dict[str, pd.DataFrame]] = []

if RUN_FULL_ROBUSTNESS:
    if not SMOKE_READY_FOR_FULL:
        raise RuntimeError(
            "Full robustness run blocked because smoke audit failed."
        )

    expected_task_count = (
        len(CONFIG.datasets)
        * len(CONFIG.k_values)
        * len(CONFIG.confirmatory_seeds)
    )

    completed = 0

    for dataset_name in CONFIG.datasets:
        bundle = BUNDLES[dataset_name]

        for k in CONFIG.k_values:
            for seed in CONFIG.confirmatory_seeds:
                task_key = make_task_key(
                    dataset_name,
                    k,
                    seed,
                )

                if (
                    dataset_name == CONFIG.smoke_dataset
                    and k == CONFIG.smoke_k
                    and seed == CONFIG.smoke_seed
                ):
                    output = SMOKE_OUTPUT
                    print(f"[REUSE] {task_key}")
                else:
                    print(f"[RUN]   {task_key}")
                    output = run_robustness_task(
                        bundle,
                        k=k,
                        seed=seed,
                        config=CONFIG,
                    )

                save_task_predictions(
                    output,
                    task_key,
                )

                ROBUSTNESS_SMALL_OUTPUTS.append(
                    small_view(output)
                )

                completed += 1

                print(
                    f"Completed: {completed} / {expected_task_count}"
                )

    save_small_bundle(
        ROBUSTNESS_SMALL_OUTPUTS,
        prefix="robustness",
    )

    print(
        f"Backbone robustness completed: "
        f"{completed}/{expected_task_count}"
    )
else:
    print(
        "Full robustness run disabled. "
        "Inspect smoke output, then set RUN_FULL_ROBUSTNESS=True."
    )


In [ ]:

# Cell 22 — Completion report and protocol audit

if RUN_FULL_ROBUSTNESS:
    robustness_results = collect_small_table(
        ROBUSTNESS_SMALL_OUTPUTS,
        "task_results",
    )

    robustness_audit = collect_small_table(
        ROBUSTNESS_SMALL_OUTPUTS,
        "protocol_audit",
    )

    expected_task_count = (
        len(CONFIG.datasets)
        * len(CONFIG.k_values)
        * len(CONFIG.confirmatory_seeds)
    )

    unique_tasks = robustness_audit["task_key"].nunique()
    all_pass = bool(
        robustness_audit["overall_protocol_pass"].all()
    )

    completion_report = {
        "expected_tasks": expected_task_count,
        "completed_unique_tasks": int(unique_tasks),
        "all_protocol_pass": all_pass,
        "backbone": CONFIG.backbone_name,
        "weights_name": CONFIG.weights_name,
        "config_hash": CONFIG_HASH,
    }

    write_json(
        completion_report,
        OUTPUT_DIR / "robustness_completion_report.json",
    )

    print(json.dumps(completion_report, indent=2))

    if unique_tasks != expected_task_count:
        raise AssertionError(
            f"Expected {expected_task_count} tasks, "
            f"found {unique_tasks}."
        )

    if not all_pass:
        failed = robustness_audit.loc[
            ~robustness_audit["overall_protocol_pass"],
            "task_key",
        ].tolist()

        raise AssertionError(
            f"Protocol audit failed for: {failed}"
        )

    print("Backbone robustness protocol audit: PASSED")


In [ ]:

# Cell 23 — Aggregate robustness summary

if RUN_FULL_ROBUSTNESS:
    metric_columns = [
        "test_macro_f1",
        "test_accuracy",
        "test_balanced_accuracy",
        "test_nll",
        "test_ece",
        "test_brier",
    ]

    robustness_summary = (
        robustness_results
        .groupby(
            [
                "backbone",
                "dataset",
                "K",
                "method",
            ]
        )[metric_columns]
        .agg(["mean", "std"])
        .reset_index()
    )

    robustness_summary.to_csv(
        OUTPUT_DIR / "robustness_summary_mean_std.csv",
        index=False,
    )

    display(robustness_summary)


In [ ]:

# Cell 24 — Paired robustness comparisons

def paired_comparison_table(
    results: pd.DataFrame,
    method: str,
    baseline: str,
    metrics: Sequence[str],
) -> pd.DataFrame:
    rows = []

    for (dataset, k), subset in results.groupby(
        ["dataset", "K"]
    ):
        left = subset.loc[
            subset["method"] == method
        ].set_index("seed")

        right = subset.loc[
            subset["method"] == baseline
        ].set_index("seed")

        common = left.index.intersection(right.index)

        for metric in metrics:
            delta = (
                left.loc[common, metric]
                - right.loc[common, metric]
            )

            higher_is_better = metric in {
                "test_macro_f1",
                "test_accuracy",
                "test_balanced_accuracy",
            }

            if higher_is_better:
                wins = int((delta > 1e-12).sum())
                losses = int((delta < -1e-12).sum())
            else:
                wins = int((delta < -1e-12).sum())
                losses = int((delta > 1e-12).sum())

            rows.append({
                "dataset": dataset,
                "K": int(k),
                "method": method,
                "baseline": baseline,
                "metric": metric,
                "mean_signed_delta": float(delta.mean()),
                "std_signed_delta": float(delta.std(ddof=1)),
                "wins": wins,
                "ties": int((np.abs(delta) <= 1e-12).sum()),
                "losses": losses,
                "n_pairs": int(len(common)),
            })

    return pd.DataFrame(rows)


if RUN_FULL_ROBUSTNESS:
    core_metrics = [
        "test_macro_f1",
        "test_nll",
        "test_ece",
        "test_brier",
    ]

    comparison_specs = [
        ("PFG-NW-F1Only", "PFG-UnweightedLR"),
        ("CA-PFG-MO", "PFG-NW-F1Only"),
        ("CA-PFG-MO", "LR-Real"),
    ]

    paired_results = pd.concat(
        [
            paired_comparison_table(
                robustness_results,
                method,
                baseline,
                core_metrics,
            )
            for method, baseline in comparison_specs
        ],
        ignore_index=True,
    )

    paired_results.to_csv(
        OUTPUT_DIR / "robustness_paired_comparisons.csv",
        index=False,
    )

    display(paired_results)


In [ ]:

# Cell 25 — Final compact package

summary_zip = create_summary_zip(
    "CAPFG11_backbone_robustness_summary_package.zip"
)

print("Summary ZIP:", summary_zip)

print("\nFeature cache files:")
for path in sorted(FEATURE_CACHE_DIR.iterdir()):
    print(" -", path.name)

print("\nTop-level result files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

print(
    "\nNote: the summary ZIP excludes ResNet feature caches "
    "and per-image prediction files."
)



## Pre-specified robustness interpretation

Notebook 11 evaluates whether the following confirmatory conclusions reproduce under a second backbone.

### R1 — normalized synthetic budget

Compare:

```text
PFG-NW-F1Only
vs
PFG-UnweightedLR
```

### R2 — calibration-aware model selection

Compare:

```text
CA-PFG-MO
vs
PFG-NW-F1Only
```

### R3 — competitive classification with improved calibration

Compare:

```text
CA-PFG-MO
vs
LR-Real
```

The robustness experiment is supportive evidence. It does not redefine or retune the frozen CA-PFG method.
